In [1]:
!pip uninstall -y jax jaxlib ml-dtypes tensorflow tf-keras keras
!pip install -q ml-dtypes==0.5.1
!pip install -q tensorflow==2.16.1 tf-keras~=2.16 opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 23.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.16.1 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.16.1 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tf_keras~=2.19, but you have tf-keras 2.16.0 which is incompatible.
dopamine-rl 4.1.2 requires tf-keras>=2.18.0, but you have tf-keras 2.16.0 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not current

In [8]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tensorflow import keras
import numpy as np
import cv2


emnist_labels = [48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122]

def emnist_predict_img(model, img):
    img_arr = np.expand_dims(img, axis=0)
    img_arr = 1 - img_arr / 255.0
    img_arr[0] = np.rot90(img_arr[0], 3)
    img_arr[0] = np.fliplr(img_arr[0])
    img_arr = img_arr.reshape((1, 28, 28, 1))
    result = np.argmax(model.predict(img_arr, verbose=0), axis=1)
    return chr(emnist_labels[result[0]])

def img_to_str(model, letters):
    result = ""
    for i in range(len(letters)):
        dn = letters[i + 1][0] - letters[i][0] - letters[i][1] if i < len(letters) - 1 else 0
        result += emnist_predict_img(model, letters[i][2])
        if dn > letters[i][1] / 4:
            result += " "
    return result

image_path = "/content/image.png"
model_path = "/content/emnist_symbols.h5"

img = cv2.imread(image_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

thresh = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
)

contours, h = cv2.findContours(thresh, 1, 2)

largest_rectangle = [0, None, None]
for cnt in contours:
    approx = cv2.approxPolyDP(cnt, 0.01 * cv2.arcLength(cnt, True), True)
    if len(approx) == 4:
        area = cv2.contourArea(cnt)
        if area > largest_rectangle[0]:
            largest_rectangle = [area, cnt, approx]

x, y, w, h = cv2.boundingRect(largest_rectangle[1])
vehicle_num = img[y:y + h, x:x + w]

gray = cv2.cvtColor(vehicle_num, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 75, 255, cv2.THRESH_BINARY)
img_erode = cv2.erode(thresh, np.ones((3, 3), np.uint8), iterations=1)

contours, hierarchy = cv2.findContours(img_erode, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)

letters = []
for idx, contour in enumerate(contours):
    x, y, w, h = cv2.boundingRect(contour)
    if w > 10 and h > 10:
        if hierarchy[0][idx][3] == 0:
            letter_crop = gray[y:y + h, x:x + w]
            size_max = max(w, h)
            letter_square = 255 * np.ones((size_max, size_max), dtype=np.uint8)

            if w > h:
                y_pos = size_max // 2 - h // 2
                letter_square[y_pos:y_pos + h, 0:w] = letter_crop
            elif w < h:
                x_pos = size_max // 2 - w // 2
                letter_square[0:h, x_pos:x_pos + w] = letter_crop
            else:
                letter_square = letter_crop

            letter_resized = cv2.resize(letter_square, (28, 28), interpolation=cv2.INTER_AREA)
            letters.append((x, w, letter_resized))

letters.sort(key=lambda x: x[0])

model = keras.models.load_model(model_path, compile=False)
result = img_to_str(model, letters)

print("raw:", result)
print("answer:", result.replace(" ", "").lower())

raw: B777PT
answer: b777pt
